# Waveform for one HLC hit — charge vs. time

Loads `one_event_waveforms.pkl` (run `extract_one_event_waveforms.py` to regenerate).

**Fundamental data:** the DOM samples the PMT signal at fixed intervals.
Each sample = ladning indsamlet i den bin.

- **ATWD ch0:** 128 samples × 3.33 ns/bin  →  høj tidsopløsning, ~420 ns dækning
- **fADC:**     256 samples × 25 ns/bin    →  lavere opløsning, ~6.4 µs dækning

Begge er målinger af *samme* PMT-strøm med forskellig opløsning.

In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

PKL = Path('one_event_waveforms.pkl')
with open(PKL, 'rb') as f:
    payload = pickle.load(f)

ev = payload['event']
hlc = payload['hlc']
slc = payload['slc']
print(f"run {ev['run_id']}  event {ev['event_id']}")
print(f"HLC DOM (string, om, pmt) = {hlc['om']}")
print(f"  PMT HV = {hlc['pmt_hv_volts']:.0f} V    gain = {hlc['pmt_gain']:.2e}    R_FE = {hlc['fe_impedance_ohm']:.0f} Ω")
print(f"SLC DOM = {slc['om']}   chargestamp = {slc['chargestamp_samples']}")

In [ ]:
# Convert each calibrated voltage sample to charge in that bin (PE per sample).
#   Q_bin = V_sample · dt / (R_FE · g_PMT · e)
k = hlc['pe_per_voltsecond']  # = 1 / (R_FE · g_PMT · e), units: PE/(V·s)

waveforms = {w['source']: w for w in hlc['calibrated_waveforms']}
order = [s for s in ('ATWD', 'FADC') if s in waveforms]

fig, axes = plt.subplots(len(order), 1, figsize=(11, 3.2 * len(order)))
if len(order) == 1:
    axes = [axes]

totals = {}
for ax, src in zip(axes, order):
    w = waveforms[src]
    V = np.asarray(w['samples_volt'])     # volts at each sample
    dt_s = w['bin_width_ns'] * 1e-9       # bin width in seconds
    pe_per_bin = V * dt_s * k             # PE in each bin
    t = w['time_ns'] + np.arange(len(pe_per_bin)) * w['bin_width_ns']
    totals[src] = pe_per_bin.sum()

    ax.step(t, pe_per_bin, where='post', lw=1.5)
    ax.set_xlabel('time [ns]')
    ax.set_ylabel('charge per sample [PE]')
    ax.set_title(f"{src}  ({len(pe_per_bin)} samples × {w['bin_width_ns']:.2f} ns/bin)   "
                 f"— total {pe_per_bin.sum():.2f} PE")
    ax.grid(alpha=0.3)

fig.suptitle(f"HLC launch  DOM {hlc['om']}", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

print('Total integrated charge:')
for src, q in totals.items():
    print(f'  {src}: {q:.2f} PE')

## Sådan læser du plottet

- **x-akse:** tid i ns
- **y-akse:** ladning (i PE) der ankom i den ene ADC-sample (bin)
- **total ladning** for et tidsinterval = summen af alle y-værdier i det interval

ATWD-bar er højere end fADC-bar omkring toppen, **ikke** fordi der er mere lys —
men fordi fADC-bin'en (25 ns) midler over et bredere tidsvindue, så den
samme topværdi smøres ud. Men begges sum giver samme totale ladning (~10 PE
for dette event).

Det ér forskellen mellem HLC og SLC: HLC sender hele dette array af samples,
SLC sender kun de 3 fADC-samples omkring toppen.

In [ ]:
# --- SLC for comparison: only 3 fADC samples around the peak survive ---
cs = slc['chargestamp_samples']
peak_idx = slc['chargestamp_highest_sample']
if cs:
    rel_idx = np.array([peak_idx - 1, peak_idx, peak_idx + 1])
    t = slc['time_ns'] + rel_idx * slc['fadc_bin_ns']
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.stem(t, cs, basefmt=' ')
    ax.set_xlabel('time [ns]')
    ax.set_ylabel('raw ADC count')
    ax.set_title(f"SLC chargestamp  DOM {slc['om']}  —  3 fADC samples around peak")
    ax.grid(alpha=0.3)
    plt.show()

# Brightest HLC launch — does this one show afterpulses?

Vælg `IceTray (py3-v4.3.0 / icetray v1.11.1)` som kernel, så kan vi scanne
I3-filen direkte herfra i stedet for via et separat script.

Den brighteste HLC-DOM i de første `N_FRAMES` frames er typisk et lyst
atmosfærisk muon, hvor ion-feedback-afterpulses bliver tydelige (~1-3% af
hovedpulsens ladning, fordelt ~0.3-12 µs efter hovedpulsen, med peak omkring 6 µs).

In [ ]:
import subprocess
import textwrap
import tempfile
import pickle
import os
import sys
from pathlib import Path

ENV_SHELL = '/cvmfs/icecube.opensciencegrid.org/py3-v4.3.0/RHEL_9_x86_64/metaprojects/icetray/v1.11.1/env-shell.sh'

def run_in_icetray(python_code: str, timeout: int = 7200) -> Path:
    tmp_py  = Path(tempfile.mkstemp(suffix='.py')[1])
    tmp_pkl = Path(tempfile.mkstemp(suffix='.pkl')[1])
    tmp_py.write_text(textwrap.dedent(python_code))
    proc = subprocess.Popen(
        [ENV_SHELL, 'python', '-u', str(tmp_py)],
        env={**os.environ, 'OUT_PICKLE': str(tmp_pkl), 'PYTHONUNBUFFERED': '1'},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        for line in proc.stdout:
            print(line.rstrip())
            sys.stdout.flush()
        proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        proc.kill()
        raise
    if proc.returncode != 0:
        raise RuntimeError(f'icetray subprocess failed (rc={proc.returncode})')
    return tmp_pkl


I3_FILE  = '/lustre/hpc/icecube/janikh/MINIONS_DA_sample_2015_v5.i3.zst'
GCD_FILE = '/lustre/hpc/icecube/janikh/GeoCalibDetectorStatus_2015.57161_V0.i3.gz'

N_FRAMES         = 200000
PROGRESS_EVERY   = 2000

MAIN_WINDOW_NS   = 300.0
MAIN_PEAK_MIN    = 0.5
AP_WINDOW_NS     = (2000.0, 10000.0)
AP_PEAK_MIN      = 10.0
CONTRAST_MIN     = 10.0
EARLY_STOP_PEAK  = 12.0
EARLY_STOP_CTR   = 15.0

scan_code = f'''
    import os, math, pickle, statistics, sys, time
    from icecube import icetray, dataio, dataclasses, WaveCalibrator
    from icecube.icetray import I3Tray, I3Units

    FE_R, E_CHG = 50.0, 1.602176634e-19
    MAIN_W       = {MAIN_WINDOW_NS}
    MAIN_PEAK_MIN= {MAIN_PEAK_MIN}
    AP_LO, AP_HI = {AP_WINDOW_NS[0]}, {AP_WINDOW_NS[1]}
    AP_PEAK_MIN  = {AP_PEAK_MIN}
    CONTRAST_MIN = {CONTRAST_MIN}
    EARLY_STOP_PEAK = {EARLY_STOP_PEAK}
    EARLY_STOP_CTR  = {EARLY_STOP_CTR}
    PROGRESS_EVERY  = {PROGRESS_EVERY}
    N_FRAMES        = {N_FRAMES}

    best = {{"score": -1.0}}
    candidates = []  # all "new best" found along the way
    found = [False]
    seen  = 0
    t_start = time.time()

    def scan(frame):
        global seen, best
        seen += 1
        if seen % PROGRESS_EVERY == 0:
            elapsed = time.time() - t_start
            rate = seen / elapsed if elapsed > 0 else 0
            cur = best.get("ap_peak_pe", 0.0)
            print(f"  {{seen:>7d}} / {{N_FRAMES}} checked  "
                  f"({{elapsed:5.0f}}s, {{rate:5.0f}} frames/s)  "
                  f"best AP peak so far: {{cur:.1f}} PE", flush=True)
        if seen > N_FRAMES or found[0]: return
        if "InIceRawData" not in frame or "CalibratedWaveforms" not in frame: return
        rd, cal_wfs = frame["InIceRawData"], frame["CalibratedWaveforms"]
        cal, det = frame["I3Calibration"], frame["I3DetectorStatus"]
        for om, launches in rd:
            L = next((x for x in launches if x.lc_bit), None)
            if L is None or om not in cal_wfs: continue
            dc, ds = cal.dom_cal[om], det.dom_status[om]
            hv = float(ds.pmt_hv) / I3Units.V
            if hv <= 0: continue
            gain = 10 ** (dc.hv_gain_fit.intercept + dc.hv_gain_fit.slope * math.log10(hv))
            kf = 1.0 / (FE_R * gain * E_CHG)
            fadc = next((w for w in cal_wfs[om] if str(w.source) == "FADC"), None)
            if fadc is None: continue
            V = [float(v) / I3Units.V for v in fadc.waveform]
            dt_s = float(fadc.bin_width) * 1e-9
            pe = [v * dt_s * kf for v in V]
            t0 = float(fadc.time)
            dt_ns = float(fadc.bin_width)
            t_main = float(L.time)

            main_peak = max(
                (q for i, q in enumerate(pe)
                 if abs((t0 + i * dt_ns) - t_main) <= MAIN_W),
                default=0.0,
            )
            if main_peak < MAIN_PEAK_MIN: continue

            ap_vals = []
            ap_peak = 0.0
            ap_peak_t = None
            for i, q in enumerate(pe):
                ti = t0 + i * dt_ns
                if AP_LO <= (ti - t_main) <= AP_HI:
                    ap_vals.append(q)
                    if q > ap_peak:
                        ap_peak = q; ap_peak_t = ti
            if len(ap_vals) < 20 or ap_peak < AP_PEAK_MIN: continue

            med = statistics.median(ap_vals)
            base = max(med, 0.05)
            contrast = ap_peak / base
            if contrast < CONTRAST_MIN: continue

            score = contrast * ap_peak
            if score > best["score"]:
                hdr = frame["I3EventHeader"]
                payload = {{
                    "score": score,
                    "frame_no": seen,
                    "ap_peak_pe": ap_peak,
                    "ap_peak_t_ns": ap_peak_t,
                    "ap_median_pe": med,
                    "ap_contrast": contrast,
                    "ap_sum_pe": sum(ap_vals),
                    "q_fadc_pe": sum(pe),
                    "main_peak_pe": main_peak,
                    "om": (int(om.string), int(om.om), int(om.pmt)),
                    "pmt_gain": float(gain),
                    "pmt_hv_volts": hv,
                    "pe_per_voltsecond": kf,
                    "hlc_launch_time_ns": t_main,
                    "event": {{"run_id": hdr.run_id, "event_id": hdr.event_id}},
                    "calibrated_waveforms": [
                        {{"source": str(w.source), "channel": int(w.channel),
                          "time_ns": float(w.time), "bin_width_ns": float(w.bin_width),
                          "samples_volt": [float(v) / I3Units.V for v in w.waveform]}}
                        for w in cal_wfs[om]
                    ],
                }}
                best = payload
                candidates.append(payload)
                print(f"    -> new best at frame {{seen}}: main peak = {{main_peak:.1f}} PE, "
                      f"AP peak = {{ap_peak:.2f}} PE, contrast = {{contrast:.1f}}x  "
                      f"[run {{hdr.run_id}} event {{hdr.event_id}} DOM {{(int(om.string), int(om.om), int(om.pmt))}}]", flush=True)
                if ap_peak > EARLY_STOP_PEAK and contrast > EARLY_STOP_CTR:
                    found[0] = True; return

    def drop_existing(frame):
        for k in ("CalibratedWaveforms", "CalibrationErrata"):
            if k in frame: del frame[k]

    def stop(frame):
        if seen >= N_FRAMES or found[0]: tray.RequestSuspension()

    tray = I3Tray()
    tray.Add("I3Reader", FilenameList=["{GCD_FILE}", "{I3_FILE}"])
    tray.Add(drop_existing, Streams=[icetray.I3Frame.DAQ])
    tray.Add("I3WaveCalibrator", Launches="InIceRawData",
             Waveforms="CalibratedWaveforms",
             WaveformRange="CalibratedWaveformRange_recalc")
    tray.Add(scan, Streams=[icetray.I3Frame.DAQ])
    tray.Add(stop, Streams=[icetray.I3Frame.DAQ])
    tray.Execute()

    with open(os.environ["OUT_PICKLE"], "wb") as f:
        pickle.dump({{"best": best, "candidates": candidates}},
                    f, protocol=pickle.HIGHEST_PROTOCOL)

    print()
    print("=" * 70)
    print("ALL candidates seen along the way (most recent = final best):")
    print(f"  {{'frame':>7s}}  {{'run':>7s}}  {{'event':>10s}}  {{'DOM':>15s}}  "
          f"{{'main':>6s}}  {{'AP':>6s}}  {{'ctr':>7s}}")
    for c in candidates:
        print(f"  {{c['frame_no']:>7d}}  {{c['event']['run_id']:>7d}}  "
              f"{{c['event']['event_id']:>10d}}  {{str(c['om']):>15s}}  "
              f"{{c['main_peak_pe']:>6.1f}}  {{c['ap_peak_pe']:>6.2f}}  "
              f"{{c['ap_contrast']:>6.1f}}x")
'''

pkl_path = run_in_icetray(scan_code)
with open(pkl_path, 'rb') as f:
    result = pickle.load(f)
best = result['best']
candidates = result['candidates']
print(f"\n→ {len(candidates)} candidates kept in `candidates` list.")
print(f"→ `best` is currently the final one. Use the next cell to load any other.")

In [ ]:
# Pick a specific (run, event, DOM) from the candidates list above.
# Either edit these constants by hand, or grab from `candidates` programmatically.

TARGET_RUN   = 126491    # e.g. 126385
TARGET_EVENT = 30343391    # e.g. 72302637
TARGET_OM    = (83, 31, 0)    # e.g. (38, 13, 0)   — set to None to take any DOM in that event

# Quick shortcut: pick by candidate index (0-based; None means "use targets above")
PICK_INDEX   = None    # e.g. 3  to load candidates[3]

if PICK_INDEX is not None:
    best = candidates[PICK_INDEX]
    print(f"Loaded candidates[{PICK_INDEX}]: "
          f"run {best['event']['run_id']} event {best['event']['event_id']} DOM {best['om']}")
elif TARGET_RUN is not None and TARGET_EVENT is not None:
    # Try first the in-memory candidates list (no I3 read needed)
    hit = next(
        (c for c in candidates
         if c['event']['run_id'] == TARGET_RUN
         and c['event']['event_id'] == TARGET_EVENT
         and (TARGET_OM is None or c['om'] == TARGET_OM)),
        None,
    )
    if hit is not None:
        best = hit
        print(f"Loaded from candidates: run {TARGET_RUN} event {TARGET_EVENT} DOM {best['om']}")
    else:
        # Fall back: re-read the I3 file looking only for this (run, event, DOM)
        print(f"Not in candidates list — fetching from I3 file ...")
        fetch_code = f'''
            import os, math, pickle
            from icecube import icetray, dataio, dataclasses, WaveCalibrator
            from icecube.icetray import I3Tray, I3Units

            FE_R, E_CHG = 50.0, 1.602176634e-19
            TARGET_RUN, TARGET_EVENT = {TARGET_RUN}, {TARGET_EVENT}
            TARGET_OM = {TARGET_OM!r}
            out = [None]

            def grab(frame):
                if out[0]: return
                if "InIceRawData" not in frame or "CalibratedWaveforms" not in frame: return
                hdr = frame["I3EventHeader"]
                if hdr.run_id != TARGET_RUN or hdr.event_id != TARGET_EVENT: return
                rd, cal_wfs = frame["InIceRawData"], frame["CalibratedWaveforms"]
                cal, det = frame["I3Calibration"], frame["I3DetectorStatus"]
                for om, launches in rd:
                    om_t = (int(om.string), int(om.om), int(om.pmt))
                    if TARGET_OM is not None and om_t != TARGET_OM: continue
                    L = next((x for x in launches if x.lc_bit), None)
                    if L is None or om not in cal_wfs: continue
                    dc, ds = cal.dom_cal[om], det.dom_status[om]
                    hv = float(ds.pmt_hv) / I3Units.V
                    gain = 10 ** (dc.hv_gain_fit.intercept + dc.hv_gain_fit.slope * math.log10(hv))
                    kf = 1.0 / (FE_R * gain * E_CHG)
                    out[0] = {{
                        "om": om_t,
                        "pmt_gain": float(gain),
                        "pmt_hv_volts": hv,
                        "pe_per_voltsecond": kf,
                        "hlc_launch_time_ns": float(L.time),
                        "event": {{"run_id": hdr.run_id, "event_id": hdr.event_id}},
                        "calibrated_waveforms": [
                            {{"source": str(w.source), "channel": int(w.channel),
                              "time_ns": float(w.time), "bin_width_ns": float(w.bin_width),
                              "samples_volt": [float(v) / I3Units.V for v in w.waveform]}}
                            for w in cal_wfs[om]
                        ],
                    }}
                    break

            def drop_existing(frame):
                for k in ("CalibratedWaveforms", "CalibrationErrata"):
                    if k in frame: del frame[k]

            def stop(frame):
                if out[0]: tray.RequestSuspension()

            tray = I3Tray()
            tray.Add("I3Reader", FilenameList=["{GCD_FILE}", "{I3_FILE}"])
            tray.Add(drop_existing, Streams=[icetray.I3Frame.DAQ])
            tray.Add("I3WaveCalibrator", Launches="InIceRawData",
                     Waveforms="CalibratedWaveforms",
                     WaveformRange="CalibratedWaveformRange_recalc")
            tray.Add(grab, Streams=[icetray.I3Frame.DAQ])
            tray.Add(stop, Streams=[icetray.I3Frame.DAQ])
            tray.Execute()

            if out[0] is None:
                raise RuntimeError("event/DOM not found in file")
            with open(os.environ["OUT_PICKLE"], "wb") as f:
                pickle.dump(out[0], f, protocol=pickle.HIGHEST_PROTOCOL)
            print(f"fetched DOM {{out[0]['om']}}")
        '''
        fp = run_in_icetray(fetch_code)
        with open(fp, 'rb') as f:
            best = pickle.load(f)
else:
    print("No target set — `best` is unchanged (still the scan's final candidate).")

In [ ]:
bright = best   # alias

print(f"DOM {bright['om']}   total fADC charge = {bright['q_fadc_pe']:.1f} PE   "
      f"gain={bright['pmt_gain']:.2e}")

k = bright['pe_per_voltsecond']
waveforms = {w['source']: w for w in bright['calibrated_waveforms']}
order = [s for s in ('ATWD', 'FADC') if s in waveforms]

fig, axes = plt.subplots(len(order), 1, figsize=(11, 3.2 * len(order)))
if len(order) == 1:
    axes = [axes]

for ax, src in zip(axes, order):
    w = waveforms[src]
    V = np.asarray(w['samples_volt'])
    dt_s = w['bin_width_ns'] * 1e-9
    pe = V * dt_s * k
    t = w['time_ns'] + np.arange(len(pe)) * w['bin_width_ns']
    ax.step(t, pe, where='post', lw=1.2)
    ax.set_xlabel('time [ns]')
    ax.set_ylabel('charge per sample [PE]')
    ax.set_title(f"{src}  ({len(pe)} samples × {w['bin_width_ns']:.2f} ns/bin)   total {pe.sum():.1f} PE")
    ax.grid(alpha=0.3)
    if src == 'FADC':
        t_main = bright['hlc_launch_time_ns']
        ax.axvspan(t_main + 300, t_main + 12000, color='orange', alpha=0.12,
                   label='afterpulse window (0.3-12 µs)')
        ax.legend(loc='upper right', fontsize=9)

fig.suptitle(f"Brightest HLC launch  DOM {bright['om']}", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Replot and save the figure with the event id in the filename.
k = best['pe_per_voltsecond']
waveforms = {w['source']: w for w in best['calibrated_waveforms']}
order = [s for s in ('ATWD', 'FADC') if s in waveforms]

fig, axes = plt.subplots(len(order), 1, figsize=(11, 3.2 * len(order)))
if len(order) == 1:
    axes = [axes]

for ax, src in zip(axes, order):
    w = waveforms[src]
    V = np.asarray(w['samples_volt'])
    dt_s = w['bin_width_ns'] * 1e-9
    pe = V * dt_s * k
    t = w['time_ns'] + np.arange(len(pe)) * w['bin_width_ns']
    ax.step(t, pe, where='post', lw=1.2)
    ax.set_xlabel('time [ns]')
    ax.set_ylabel('charge per sample [PE]')
    ax.set_title(f"{src}  ({len(pe)} samples × {w['bin_width_ns']:.2f} ns/bin)   total {pe.sum():.1f} PE")
    ax.grid(alpha=0.3)
    if src == 'FADC':
        t_main = best['hlc_launch_time_ns']
        ax.axvspan(t_main + 300, t_main + 12000, color='orange', alpha=0.12,
                   label='afterpulse window (0.3-12 µs)')
        ax.legend(loc='upper right', fontsize=9)

run = best['event']['run_id']
eid = best['event']['event_id']
fig.suptitle(f"Afterpulse example  —  run {run}  event {eid}  —  DOM {best['om']}",
             fontsize=12, y=1.02)
fig.tight_layout()

fname = f"afterpulse_run{run}_event{eid}.png"
fig.savefig(fname, dpi=140, bbox_inches='tight')
plt.show()
print(f"saved: {fname}")
print(f"event:  run={run}  id={eid}   DOM={best['om']}")

### Saved afterpulse example

Et pænt afterpulse-eksempel fra `MINIONS_DA_sample_2015_v5.i3.zst` — fundet
ved scan med kontrast-kriterium (peak ÷ median i 2-10 µs vinduet,
hovedpuls 100-500 PE).

Eksempel: **run 126385, event 72302637**.

Plottet gemmes som PNG med event-id i filnavnet (se cell-output).